# Learning Environmental Dynamics: Building Internal Models of a Simulated External Force

### **Abstract**

Motor adaptation is typically studied using limb perturbations in highly constrained tasks. However, everyday actions require compensating for uncertain environmental dynamics impacting manipulated objects within redundant execution spaces, where multiple combinations of motor variables can achieve success. To investigate this, we designed a novel virtual reality task where participants launched a ball across a lateral water current to spatially distinct targets. Two experiments (2.0 m/s and 3.0 m/s) tested if classic adaptation signatures remain consistent across varying perturbation sizes.  

Results revealed three key signatures. First, participants exhibited robust error reduction to all targets during training. Second, between-subjects analyses revealed that this learning produced substantial, immediate transfer to untrained target locations. Finally, persistent motor aftereffects appeared upon perturbation removal, despite explicit cues indicating absence of the perturbation. Together, these results demonstrate that humans form internal models under altered projectile–environment dynamics, and that prior experience facilitates broad generalization across the workspace. 

# 

# Load Data & Formatting

In [35]:
# import key libraries
library(emmeans)
library(dplyr)
library(afex)
library(effsize)

# full dataset

# set working directory & load FULL data from local storage
setwd("C:/Users/jacob/water_current_MA/data/dual_generalization")
df_full <- read.csv("PCA_error_projectile_experiment _S1-projectile_experiment_S2-projectile_experiment_S1_3.0M-projectile_experiment 1_S2_3.0M-projectile_experiment 1_speed_neg3_0_neg2_0_0_0.csv")


# Format data types 
df_full$ppid_full <- factor(df_full$ppid_full)
df_full$speed_label <- factor(df_full$speed_label)
df_full$target_x_label <- factor(df_full$target_x_label)
df_full$phase <- factor(df_full$phase)

df_full$set_order <- factor(df_full$set_order)
# change set_order names to reflect thesis, group 1 and group 2 labels
levels(df_full$set_order) <- c("group_1","group_2")

# ensure consistent levels
df_full$target_x_label <- factor(df_full$target_x_label, 
                                 levels = c("L60", "L30", "R30", "R60"))


# load the smaller, early late (first and last 2 trials/phase) dataset
df_early_late_2 <- read.csv("early_late_2_phase_PCA_error_projectile_experiment _S1-projectile_experiment_etc_speed_neg3_0_neg2_0.csv")

# Format data types 
df_early_late_2$ppid_full <- factor(df_early_late_2$ppid_full)
df_early_late_2$speed_label <- factor(df_early_late_2$speed_label)
df_early_late_2$target_x_label <- factor(df_early_late_2$target_x_label)
df_early_late_2$phase <- factor(df_early_late_2$phase)

df_early_late_2$set_order <- factor(df_early_late_2$set_order)
# change set_order names to reflect thesis, group 1 and group 2 labels
levels(df_early_late_2$set_order) <- c("group_1","group_2")

# ensure consistent levels
df_early_late_2$target_x_label <- factor(df_early_late_2$target_x_label, 
                                     levels = c("L60", "L30", "R30", "R60"))

# Overall learning (early vs late)

TRAINING: First and Last 2 trials

Here we compared the mean minimum metric error for the first and last two trials per target during the naive, initial Training Phase 1. We conducted within-subjects ANOVAs on minimum metric error with trial set and target as within-subject factors. Each ANOVA was conducted across two target pair groups (i.e., Group 1 and Group 2) and separately for two water current experiments. Consequently, we ran a total of 4 within-subjects ANOVAs (two groups, two experiments). Holm's correction was reported for the two group ANOVAs in each separate experiment. The purpose of this analysis was to examine robust learning following practice compensating for the water current perturbation.


Speed 3.0 m/s

The interaction was not significant across either group, suggesting that both targets within each pair produced comparable learning magnitudes from early to late training. However, the main effect of trial set was highly significant for both Group 1 and Group 2 after applying Holm's correction, producing large generalized eta-squared values of 0.251 and 0.301, respectively. This demonstrates a robust error reduction following practice compensating for the water current that was consistent across both target pair groups.


Speed 2.0 m/s

Crucially, the interaction in this experiment was significant in Group 1, but became a non-significant trend following Holm's correction. Nonetheless, here we also observed significant main effects of trial set for both groups after applying Holm's correction. Effect sizes were similar to the previous experiment, with Group 1 yielding a large generalized eta-squared value of 0.283, while Group 2 showed an even larger generalized eta-squared value of 0.310. Again, we observed a significant learning effect following practice in compensating for the water current.

In [45]:
# Isolate training phase
df_early_late_2_t1 <- df_early_late_2[df_early_late_2$phase == 'training_1',]
speeds_list <- unique(df_early_late_2_t1$speed_label)
set_order_list <- unique(df_early_late_2_t1$set_order)


for (speed in speeds_list) {
    
    results_storage <- list()
    raw_interaction_p_vals <- c()
    raw_trial_set_p_vals <- c()
    comparison_labels <- c()
        
    for (group in set_order_list) {
        
        df_subset <- df_early_late_2_t1[df_early_late_2_t1$speed_label == speed & df_early_late_2_t1$set_order == group, ]
        df_subset <- droplevels(df_subset) 
        
        model_anova <- aov_ez(
            id = "ppid_full",              
            dv = "flip_min_distance_xPCA_mean_bc",                  
            data = df_subset,                
            within = c("target_x_label", "trial_set"),
            fun_aggregate = mean 
        )
        
        label <- paste(speed, group)
        results_storage[[label]] <- model_anova
        
        raw_interaction_p_vals <- c(raw_interaction_p_vals, model_anova$anova_table["target_x_label:trial_set", "Pr(>F)"])
        raw_trial_set_p_vals <- c(raw_trial_set_p_vals, model_anova$anova_table["trial_set", "Pr(>F)"])
        comparison_labels <- c(comparison_labels, label)
        
        } 
            
            # Apply Holm corrections 
            adjusted_interaction_p_vals <- p.adjust(raw_interaction_p_vals, method = "holm")
            adjusted_trial_set_p_vals <- p.adjust(raw_trial_set_p_vals, method = "holm")
            
            names(adjusted_interaction_p_vals) <- comparison_labels
            names(adjusted_trial_set_p_vals) <- comparison_labels
        
            # POST-HOCS AND PRINTING
            for (label in comparison_labels) {
                
                model_anova <- results_storage[[label]]
                p_adj_inter <- adjusted_interaction_p_vals[label]
                p_adj_main <- adjusted_trial_set_p_vals[label]
                
                cat("\n========================================\n")
                cat("ANALYSIS FOR SPEED x GROUP:", label, "\n")
                cat("Holm-Adjusted Interaction p-value: ", p_adj_inter, "\n")
                cat("Holm-Adjusted trial_set Main Effect p-value: ", p_adj_main, "\n")
                cat("========================================\n")
                
                print(model_anova)
                
                # Conditional logic based on Holm-corrected results
                if (p_adj_inter <= 0.05) {
                    cat("\nSignificant Interaction. Running target-specific post-hocs:\n")
                    print(pairs(emmeans(model_anova, ~ trial_set | target_x_label), adjust="holm"))
                    
                } else if (p_adj_main <= 0.05) {
                    cat("\nNon-Sig Interaction. Main Effect of trial_set:\n")
                    print(pairs(emmeans(model_anova, ~ trial_set)))
                    
                } else {
                    cat("\nNeither effect survived Holm correction. Skipping post-hocs.\n")
                }
            } 
            
        }


ANALYSIS FOR SPEED x GROUP: -3 group_2 
Holm-Adjusted Interaction p-value:  0.2546359 
Holm-Adjusted trial_set Main Effect p-value:  1.580026e-06 
Anova Table (Type 3 tests)

Response: flip_min_distance_xPCA_mean_bc
                    Effect    df    MSE         F  ges p.value
1           target_x_label 1, 19 356.99 54.07 *** .280   <.001
2                trial_set 1, 19 415.47 51.68 *** .301   <.001
3 target_x_label:trial_set 1, 19 297.74      1.38 .008    .255
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '+' 0.1 ' ' 1

Non-Sig Interaction. Main Effect of trial_set:
 contrast     estimate   SE df t.ratio p.value
 early - late     32.8 4.56 19   7.189  <.0001

Results are averaged over the levels of: target_x_label 

ANALYSIS FOR SPEED x GROUP: -3 group_1 
Holm-Adjusted Interaction p-value:  0.1703152 
Holm-Adjusted trial_set Main Effect p-value:  3.865291e-06 
Anova Table (Type 3 tests)

Response: flip_min_distance_xPCA_mean_bc
                    Effect    df    MSE        

Transfer: early and late 2 trials

Similar to the analysis above, we conducted a within-subjects ANOVA on mean minimum metric error for the first and last two trials per target, but now during the second training phase. Again, we conducted four within-subjects ANOVAs on minimum metric error with trial set and target as within-subject factors, separately for both water current experiments. Holm's correction was also reported for the two group ANOVAs in each separate experiment. The purpose of this analysis was to examine any ongoing learning after completing Training Phase 1.


Speed 3.0 m/s

The interaction was not significant across either group, suggesting that both novel targets within each pair produced comparable learning magnitudes across this phase. Importantly, because the main effect of trial set was not significant, this demonstrates that learning had saturated by the onset of this phase. This hints at a transfer effect, where learning in the first training phase results in error saturation for novel targets in the second phase. Overall, because we did not observe a main effect or interaction involving trial set during Training Phase 2 for this experiment, this illustrates learning saturation, and thus no ongoing learning was detected.


Speed 2.0 m/s

Interestingly, the interaction in this experiment was significant for both groups. Post-hoc analyses revealed that this effect was driven by the more challenging, upstream rightward targets. However, only the R60 target in Group 2 remained significant following the post-hocs. This demonstrates a continuation of learning for arguably the most challenging target in this task, in this group. Because participants who experienced this target during the second training phase must have switched from R30 in Training Phase 1, this ongoing learning effect could be driven by transitioning from an easier to a more challenging target. Overall, because the majority of main effects or interactions involving trial set were not significant—besides R60 in Group 2—this demonstrates that, like the other experiment, learning had mostly saturated by the onset of this phase.


In [53]:
# Isolate training phase
df_early_late_2_t2 <- df_early_late_2[df_early_late_2$phase == 'training_2',]
speeds_list <- unique(df_early_late_2_t2$speed_label)
set_order_list <- unique(df_early_late_2_t2$set_order)


for (speed in speeds_list) {
    
    results_storage <- list()
    raw_interaction_p_vals <- c()
    raw_trial_set_p_vals <- c()
    comparison_labels <- c()
        
    for (group in set_order_list) {
        
        df_subset <- df_early_late_2_t2[df_early_late_2_t2$speed_label == speed & df_early_late_2_t2$set_order == group, ]
        df_subset <- droplevels(df_subset) 
        
        model_anova <- aov_ez(
            id = "ppid_full",              
            dv = "flip_min_distance_xPCA_mean_bc",                  
            data = df_subset,                
            within = c("target_x_label", "trial_set"),
            fun_aggregate = mean 
        )
        
        label <- paste(speed, group)
        results_storage[[label]] <- model_anova
        
        raw_interaction_p_vals <- c(raw_interaction_p_vals, model_anova$anova_table["target_x_label:trial_set", "Pr(>F)"])
        raw_trial_set_p_vals <- c(raw_trial_set_p_vals, model_anova$anova_table["trial_set", "Pr(>F)"])
        comparison_labels <- c(comparison_labels, label)
        
        } 
            
            # Apply Holm corrections 
            adjusted_interaction_p_vals <- p.adjust(raw_interaction_p_vals, method = "holm")
            adjusted_trial_set_p_vals <- p.adjust(raw_trial_set_p_vals, method = "holm")
            
            names(adjusted_interaction_p_vals) <- comparison_labels
            names(adjusted_trial_set_p_vals) <- comparison_labels
        
            # POST-HOCS AND PRINTING
            for (label in comparison_labels) {
                
                model_anova <- results_storage[[label]]
                p_adj_inter <- adjusted_interaction_p_vals[label]
                p_adj_main <- adjusted_trial_set_p_vals[label]
                
                cat("\n========================================\n")
                cat("ANALYSIS FOR SPEED x GROUP:", label, "\n")
                cat("Holm-Adjusted Interaction p-value: ", p_adj_inter, "\n")
                cat("Holm-Adjusted trial_set Main Effect p-value: ", p_adj_main, "\n")
                cat("========================================\n")
                
                print(model_anova)
                
                # Conditional logic based on Holm-corrected results
                if (p_adj_inter <= 0.05) {
                    cat("\nSignificant Interaction. Running target-specific post-hocs:\n")
                    print(pairs(emmeans(model_anova, ~ trial_set | target_x_label), adjust="holm"))
                    
                } else if (p_adj_main <= 0.05) {
                    cat("\nNon-Sig Interaction. Main Effect of trial_set:\n")
                    print(pairs(emmeans(model_anova, ~ trial_set)))
                    
                } else {
                    cat("\nNeither effect survived Holm correction. Skipping post-hocs.\n")
                }
            } 
            
        }


ANALYSIS FOR SPEED x GROUP: -3 group_2 
Holm-Adjusted Interaction p-value:  0.3588199 
Holm-Adjusted trial_set Main Effect p-value:  0.4562133 
Anova Table (Type 3 tests)

Response: flip_min_distance_xPCA_mean_bc
                    Effect    df    MSE         F  ges p.value
1           target_x_label 1, 19 394.87 39.58 *** .266   <.001
2                trial_set 1, 19 547.66      1.55 .019    .228
3 target_x_label:trial_set 1, 19 246.41      1.67 .009    .212
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '+' 0.1 ' ' 1

Neither effect survived Holm correction. Skipping post-hocs.

ANALYSIS FOR SPEED x GROUP: -3 group_1 
Holm-Adjusted Interaction p-value:  0.3588199 
Holm-Adjusted trial_set Main Effect p-value:  0.9759367 
Anova Table (Type 3 tests)

Response: flip_min_distance_xPCA_mean_bc
                    Effect    df    MSE         F   ges p.value
1           target_x_label 1, 24 396.26 28.61 ***  .223   <.001
2                trial_set 1, 24 194.89      0.00 <.001    .976

# Between-Subjects Transfer

In [49]:

# Filter for Transfer (Early trials only)
df_transfer <- df_early_late_2[df_early_late_2$trial_set == 'early',]
df_transfer$phase <- factor(df_transfer$phase, levels = c("training_1", "training_2"))

speeds_list <- unique(df_transfer$speed_label)
target_list <- unique(df_transfer$target_x_label)



for (s in speeds_list) {

    p_vals <- c()
    
    for (t in target_list) {
  
          cat("\n--- RUNNING BETWEEN TRANSFER ANALYSIS FOR SPEED x TARGET:", s, t, "---\n")
          
          df_subset <- df_transfer[df_transfer$speed_label == s & df_transfer$target_x_label == t,]
          df_subset <- droplevels(df_subset) 
          
          res <- t.test(flip_min_distance_xPCA_mean_bc ~ phase, data = df_subset, alternative = "greater")
          print(res)

          # show sd
          print(tapply(df_subset$flip_min_distance_xPCA_mean_bc, df_subset$phase, sd, na.rm = TRUE))
        
          # effect sizes
          print(cohen.d(flip_min_distance_xPCA_mean_bc ~ phase, data = df_subset))
        
          # Grab p-value & store
          p_vals <- c(p_vals, res$p.value)


  

    }

    print(s)
    
    # show p-vals
    print(p_vals)
    
    # adjusted p-vals
    adj_p <- p.adjust(p_vals, method = "holm")
    print(adj_p)
}



--- RUNNING BETWEEN TRANSFER ANALYSIS FOR SPEED x TARGET: -3 L60 ---

	Welch Two Sample t-test

data:  flip_min_distance_xPCA_mean_bc by phase
t = 5.2331, df = 69.244, p-value = 8.472e-07
alternative hypothesis: true difference in means between group training_1 and group training_2 is greater than 0
95 percent confidence interval:
 23.38442      Inf
sample estimates:
mean in group training_1 mean in group training_2 
               32.460245                -1.856991 

training_1 training_2 
  34.74310   25.32495 

Cohen's d

d estimate: 1.148978 (large)
95 percent confidence interval:
    lower     upper 
0.6943525 1.6036038 


--- RUNNING BETWEEN TRANSFER ANALYSIS FOR SPEED x TARGET: -3 L30 ---

	Welch Two Sample t-test

data:  flip_min_distance_xPCA_mean_bc by phase
t = 2.4853, df = 82.94, p-value = 0.007476
alternative hypothesis: true difference in means between group training_1 and group training_2 is greater than 0
95 percent confidence interval:
 5.945159      Inf
sample estima

Here we compare the mean extent error along primary axis for the first four trials per target (seperate for each water speed condition) across T1 AND T2. This analysis compares early performance in T1 (naive state) to early performance in T2 (experienced state) at identical target locations. 

Significant error reductions in T2 demonstrates transfer, where experienced participants successfully apply a previously learned compensation to a novel context (i.e., target). 

In [ ]:
# launch dev diffs

In [15]:

# track target transitions across phases based on set order
df_early_late_2 <- df_early_late_2 %>%
  mutate(
    target_track = case_when(
      set_order == "63_36" & target_x_label %in% c("L60", "L30") ~ "L60_to_L30",
      set_order == "63_36" & target_x_label %in% c("R30", "R60") ~ "R30_to_R60", 
      set_order == "36_63" & target_x_label %in% c("L30", "L60") ~ "L30_to_L60",
      set_order == "36_63" & target_x_label %in% c("R60", "R30") ~ "R60_to_R30", 
      TRUE ~ NA_character_
    )
  )

# ensure phase is factor with order
df_early_late_2$phase <- factor(df_early_late_2$phase, levels = c("training_1", "training_2"))

# Masking to late T1, early T2
df_early_late_2$mask_keep <- (df_early_late_2$phase == 'training_1' & df_early_late_2$trial_set == 'late') | 
                             (df_early_late_2$phase == 'training_2' & df_early_late_2$trial_set == 'early')
# Apply mask
df_filtered <- df_early_late_2[df_early_late_2$mask_keep, ]

# Setup variables
speeds_list <- unique(df_filtered$speed_label)
target_transition_list <- c("L60_to_L30", "R30_to_R60", "L30_to_L60", "R60_to_R30")

# SPEED LOOP
for (speed in speeds_list) {
    
    cat("\nRUNNING SWAP TRANSITIONS FOR SPEED:", speed, "\n")
    
    # Reset storage vectors
    results_storage <- list()
    raw_p_values <- c()
    comparison_labels <- c()

    # Target swap loop
    for (swap in target_transition_list) {
        
        
        df_subset <- df_filtered[df_filtered$speed_label == speed & df_filtered$target_track == swap, ]
        df_subset <- droplevels(df_subset) 
        
        model_anova <- aov_ez(
            id = "ppid_full",              
            dv = "launch_deviation",                  
            data = df_subset,                
            within = "phase", 
            fun_aggregate = mean 
        )
        
        label <- paste(speed, "|", swap)
        results_storage[[label]] <- model_anova
        
        raw_p_values <- c(raw_p_values, model_anova$anova_table["phase", "Pr(>F)"])
        comparison_labels <- c(comparison_labels, label)
        
    } 
    
    # Apply Holm correction to the 4 target swaps
    adjusted_p_values <- p.adjust(raw_p_values, method = "holm")
    names(adjusted_p_values) <- comparison_labels
    
    # POST-HOCS AND PRINTING
    for (label in comparison_labels) {
        
        model_anova <- results_storage[[label]]
        p_adj <- adjusted_p_values[label]
        
        cat("\n========================================\n")
        cat("ANALYSIS FOR:", label, "\n")
        cat("Holm-Adjusted Phase p-value: ", p_adj, "\n")
        cat("========================================\n")
        
        print(model_anova)
        
        # Conditional logic based on Holm-corrected results
        if (p_adj <= 0.05) {
            cat("\nSignificant Phase Shift. Running follow-up comparisons:\n")
            print(pairs(emmeans(model_anova, ~ phase))) 
            
        } else {
            cat("\nEffect did not survive localized Holm correction. Skipping post-hocs.\n")
        }
    } 
    
}


RUNNING SWAP TRANSITIONS FOR SPEED: -3 

ANALYSIS FOR: -3 | L60_to_L30 
Holm-Adjusted Phase p-value:  0.02452428 
Anova Table (Type 3 tests)

Response: launch_deviation
  Effect    df   MSE       F  ges p.value
1  phase 1, 19 36.18 8.72 ** .032    .008
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '+' 0.1 ' ' 1

Significant Phase Shift. Running follow-up comparisons:
 contrast                estimate  SE df t.ratio p.value
 training_1 - training_2     5.62 1.9 19   2.953  0.0082


ANALYSIS FOR: -3 | R30_to_R60 
Holm-Adjusted Phase p-value:  0.06815489 
Anova Table (Type 3 tests)

Response: launch_deviation
  Effect    df   MSE      F  ges p.value
1  phase 1, 19 46.47 3.74 + .035    .068
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '+' 0.1 ' ' 1

Effect did not survive localized Holm correction. Skipping post-hocs.

ANALYSIS FOR: -3 | L30_to_L60 
Holm-Adjusted Phase p-value:  0.02452428 
Anova Table (Type 3 tests)

Response: launch_deviation
  Effect    df    MSE      F 

In [ ]:
# launch speed

In [40]:

# track target transitions across phases based on set order
df_early_late_2 <- df_early_late_2 %>%
  mutate(
    target_track = case_when(
      set_order == "63_36" & target_x_label %in% c("L60", "L30") ~ "L60_to_L30",
      set_order == "63_36" & target_x_label %in% c("R60", "R30") ~ "R60_to_R30",
      set_order == "36_63" & target_x_label %in% c("L30", "L60") ~ "L30_to_L60",
      set_order == "36_63" & target_x_label %in% c("R30", "R60") ~ "R30_to_R60",
      TRUE ~ NA_character_
    )
  )

# ensure phase is factor with order
df_early_late_2$phase <- factor(df_early_late_2$phase, levels = c("training_1", "training_2"))

# Masking to late T1, early T2
df_early_late_2$mask_keep <- (df_early_late_2$phase == 'training_1' & df_early_late_2$trial_set == 'late') | 
                             (df_early_late_2$phase == 'training_2' & df_early_late_2$trial_set == 'early')
# Apply mask
df_filtered <- df_early_late_2[df_early_late_2$mask_keep, ]

# Setup variables
speeds_list <- unique(df_filtered$speed_label)
target_transition_list <- c("L60_to_L30", "R30_to_R60", "L30_to_L60", "R60_to_R30")

# SPEED LOOP
for (speed in speeds_list) {
    
    cat("\nRUNNING SWAP TRANSITIONS FOR SPEED:", speed, "\n")
    
    # Reset storage vectors
    results_storage <- list()
    raw_p_values <- c()
    comparison_labels <- c()

    # Target swap loop
    for (swap in target_transition_list) {
        
        
        df_subset <- df_filtered[df_filtered$speed_label == speed & df_filtered$target_track == swap, ]
        df_subset <- droplevels(df_subset) 
        
        model_anova <- aov_ez(
            id = "ppid_full",              
            dv = "launch_Speed",                  
            data = df_subset,                
            within = "phase", 
            fun_aggregate = mean 
        )
        
        label <- paste(speed, "|", swap)
        results_storage[[label]] <- model_anova
        
        raw_p_values <- c(raw_p_values, model_anova$anova_table["phase", "Pr(>F)"])
        comparison_labels <- c(comparison_labels, label)
        
    } 
    
    # Apply Holm correction to the 4 target swaps
    adjusted_p_values <- p.adjust(raw_p_values, method = "holm")
    names(adjusted_p_values) <- comparison_labels
    
    # POST-HOCS AND PRINTING
    for (label in comparison_labels) {
        
        model_anova <- results_storage[[label]]
        p_adj <- adjusted_p_values[label]
        
        cat("\n========================================\n")
        cat("ANALYSIS FOR:", label, "\n")
        cat("Holm-Adjusted Phase p-value: ", p_adj, "\n")
        cat("========================================\n")
        
        print(model_anova)
        
        # Conditional logic based on Holm-corrected results
        if (p_adj <= 0.05) {
            cat("\nSignificant Phase Shift. Running follow-up comparisons:\n")
            print(pairs(emmeans(model_anova, ~ phase))) 
            
        } else {
            cat("\nEffect did not survive localized Holm correction. Skipping post-hocs.\n")
        }
    } 
    
}


RUNNING SWAP TRANSITIONS FOR SPEED: -3 

ANALYSIS FOR: -3 | L60_to_L30 
Holm-Adjusted Phase p-value (N=4):  0.9182884 
Anova Table (Type 3 tests)

Response: launch_Speed
  Effect    df  MSE    F  ges p.value
1  phase 1, 19 0.21 0.39 .007    .542
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '+' 0.1 ' ' 1

Effect did not survive localized Holm correction. Skipping post-hocs.

ANALYSIS FOR: -3 | R30_to_R60 
Holm-Adjusted Phase p-value (N=4):  0.4233555 
Anova Table (Type 3 tests)

Response: launch_Speed
  Effect    df  MSE    F  ges p.value
1  phase 1, 24 0.28 2.82 .033    .106
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '+' 0.1 ' ' 1

Effect did not survive localized Holm correction. Skipping post-hocs.

ANALYSIS FOR: -3 | L30_to_L60 
Holm-Adjusted Phase p-value (N=4):  0.9182884 
Anova Table (Type 3 tests)

Response: launch_Speed
  Effect    df  MSE    F  ges p.value
1  phase 1, 24 0.23 1.09 .017    .306
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '+' 0.1 ' ' 